# RGB + DSM/CHM Non-Musanga Crown Delineation

This notebook delineates individual **non-Musanga tree crowns** from UAV RGB and height data. It uses the existing U-Net Musanga mask to remove Musanga-dominated canopy, then applies marker-controlled watershed to RGB and canopy-height information.

**Outputs:** aligned CHM, non-Musanga canopy mask, crown markers, labelled crown raster, crown polygons (GPKG/SHP), QA figures, and optional comparison with digitized non-Musanga crowns.


## BLOCK 01 — Imports and Drive

In [ ]:

# ======================================================================================
# BLOCK 01 — IMPORTS AND DRIVE
# ======================================================================================
!pip -q install geopandas rasterio shapely scikit-image scipy pyogrio rtree

from pathlib import Path
import json, math, warnings
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from google.colab import drive
from rasterio.enums import Resampling
from rasterio.features import shapes
from rasterio.warp import reproject
from scipy import ndimage as ndi
from shapely.geometry import shape, box
from skimage import filters, morphology, segmentation
from skimage.feature import peak_local_max
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

DRIVE_ROOT = Path("/content/drive")
MYDRIVE = DRIVE_ROOT / "MyDrive"

if not MYDRIVE.exists() or not any(MYDRIVE.iterdir()):
    drive.mount(str(DRIVE_ROOT), force_remount=False)
else:
    print("Google Drive already mounted.")

PROJECT_ROOT = MYDRIVE / "Colab Notebooks" / "Musanga"
WORK_DIR = PROJECT_ROOT / "work"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = WORK_DIR / "crown_delineation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)


## BLOCK 02 — Input paths and processing settings

In [ ]:

# ======================================================================================
# BLOCK 02 — INPUT PATHS AND PROCESSING SETTINGS
# ======================================================================================

# Set paths manually if automatic discovery does not find the correct files.
RGB_PATH = Path("/content/musanga_training/sources/Carlier2020_Aug24_25_survey_transparent_mosaic_group1.tif")
DSM_PATH = None
DTM_PATH = None
MUSANGA_MASK_PATH = None
REFERENCE_CROWNS_PATH = None

SEARCH_ROOTS = [
    Path("/content/musanga_training/sources"),
    DATA_DIR,
    PROJECT_ROOT,
    WORK_DIR,
]

def find_first(patterns):
    matches = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for pattern in patterns:
            matches.extend(root.glob(pattern))
    return sorted(matches)[0] if matches else None

if DSM_PATH is None:
    DSM_PATH = find_first(["**/*DSM*.tif", "**/*dsm*.tif", "**/*surface*.tif"])

if DTM_PATH is None:
    DTM_PATH = find_first(["**/*DTM*.tif", "**/*dtm*.tif", "**/*terrain*.tif"])

if MUSANGA_MASK_PATH is None:
    MUSANGA_MASK_PATH = find_first(["**/musanga_binary_mask_threshold_*.tif"])

if REFERENCE_CROWNS_PATH is None:
    REFERENCE_CROWNS_PATH = find_first([
        "**/*tree*.shp", "**/*crown*.shp", "**/*classified*.shp",
        "**/*tree*.gpkg", "**/*crown*.gpkg",
    ])

required = {
    "RGB": RGB_PATH,
    "DSM": DSM_PATH,
    "Musanga mask": MUSANGA_MASK_PATH,
}

missing = [
    name for name, path in required.items()
    if path is None or not Path(path).exists()
]

if missing:
    raise FileNotFoundError(
        f"Missing required inputs: {missing}\n"
        "Edit the paths at the top of Block 02."
    )

RGB_PATH = Path(RGB_PATH)
DSM_PATH = Path(DSM_PATH)
DTM_PATH = Path(DTM_PATH) if DTM_PATH else None
MUSANGA_MASK_PATH = Path(MUSANGA_MASK_PATH)
REFERENCE_CROWNS_PATH = Path(REFERENCE_CROWNS_PATH) if REFERENCE_CROWNS_PATH else None

# Crown delineation does not require the original centimetre-scale grid.
TARGET_RESOLUTION_M = 0.20

# Biological / segmentation settings.
MIN_TREE_HEIGHT_M = 2.0
MIN_PEAK_HEIGHT_M = 3.0
MIN_CROWN_SPACING_M = 2.0
CHM_SMOOTHING_SIGMA_M = 0.8
MUSANGA_BUFFER_M = 1.0
MIN_CROWN_AREA_M2 = 2.0
MAX_CROWN_AREA_M2 = 400.0

print("RGB:", RGB_PATH)
print("DSM:", DSM_PATH)
print("DTM:", DTM_PATH)
print("Musanga mask:", MUSANGA_MASK_PATH)
print("Reference crowns:", REFERENCE_CROWNS_PATH)


## BLOCK 03 — Create a common working grid and CHM

In [ ]:

# ======================================================================================
# BLOCK 03 — COMMON WORKING GRID AND CHM
# ======================================================================================

with rasterio.open(RGB_PATH) as src:
    rgb_width = src.width
    rgb_height = src.height
    rgb_transform = src.transform
    rgb_crs = src.crs
    native_resolution = abs(src.transform.a)

scale = max(1, int(round(TARGET_RESOLUTION_M / native_resolution)))
work_width = math.ceil(rgb_width / scale)
work_height = math.ceil(rgb_height / scale)
work_transform = rgb_transform * rasterio.Affine.scale(scale, scale)
work_resolution = native_resolution * scale

print("Native resolution:", native_resolution)
print("Working resolution:", work_resolution)
print("Working dimensions:", work_width, "x", work_height)

def align_array(path, bands, dtype, resampling, nodata):
    destination = np.full(
        (bands, work_height, work_width),
        nodata,
        dtype=dtype,
    )
    with rasterio.open(path) as src:
        for band in range(1, bands + 1):
            reproject(
                source=rasterio.band(src, band),
                destination=destination[band - 1],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=work_transform,
                dst_crs=rgb_crs,
                resampling=resampling,
                src_nodata=src.nodata,
                dst_nodata=nodata,
            )
    return destination

rgb = align_array(
    RGB_PATH, 3, np.uint8, Resampling.bilinear, 0
)
dsm = align_array(
    DSM_PATH, 1, np.float32, Resampling.bilinear, -9999.0
)[0]
musanga_raw = align_array(
    MUSANGA_MASK_PATH, 1, np.uint8, Resampling.nearest, 255
)[0]

dsm_valid = dsm != -9999.0

if DTM_PATH is not None:
    dtm = align_array(
        DTM_PATH, 1, np.float32, Resampling.bilinear, -9999.0
    )[0]
    chm_valid = dsm_valid & (dtm != -9999.0)
    chm = np.full_like(dsm, -9999.0)
    chm[chm_valid] = dsm[chm_valid] - dtm[chm_valid]
else:
    print("No DTM found; estimating local ground from DSM.")
    fill_value = np.nanmedian(np.where(dsm_valid, dsm, np.nan))
    dsm_fill = np.where(dsm_valid, dsm, fill_value)
    window_px = max(3, int(round(20.0 / work_resolution)))
    if window_px % 2 == 0:
        window_px += 1
    local_ground = ndi.percentile_filter(
        dsm_fill, percentile=5, size=window_px, mode="nearest"
    )
    chm_valid = dsm_valid
    chm = np.full_like(dsm, -9999.0)
    chm[chm_valid] = dsm[chm_valid] - local_ground[chm_valid]

chm[chm_valid] = np.clip(chm[chm_valid], 0, None)

CHM_PATH = OUTPUT_DIR / "chm_working_grid.tif"

profile = {
    "driver": "GTiff",
    "height": work_height,
    "width": work_width,
    "count": 1,
    "dtype": "float32",
    "crs": rgb_crs,
    "transform": work_transform,
    "nodata": -9999.0,
    "compress": "LZW",
    "tiled": True,
    "blockxsize": 512,
    "blockysize": 512,
    "BIGTIFF": "YES",
}

with rasterio.open(CHM_PATH, "w", **profile) as dst:
    dst.write(chm, 1)

print("Saved CHM:", CHM_PATH)


## BLOCK 04 — Build non-Musanga canopy mask

In [ ]:

# ======================================================================================
# BLOCK 04 — NON-MUSANGA CANOPY MASK
# ======================================================================================

rgb_hwc = np.moveaxis(rgb, 0, -1).astype(np.float32) / 255.0
rgb_valid = np.any(rgb_hwc > 0, axis=2)

# Simple RGB vegetation constraint.
excess_green = (
    2 * rgb_hwc[..., 1]
    - rgb_hwc[..., 0]
    - rgb_hwc[..., 2]
)
vegetation = excess_green > -0.10

musanga = musanga_raw == 1
buffer_px = max(0, int(round(MUSANGA_BUFFER_M / work_resolution)))

if buffer_px > 0:
    musanga_exclusion = morphology.binary_dilation(
        musanga,
        morphology.disk(buffer_px),
    )
else:
    musanga_exclusion = musanga

canopy_mask = (
    rgb_valid
    & chm_valid
    & vegetation
    & (chm >= MIN_TREE_HEIGHT_M)
    & ~musanga_exclusion
)

minimum_fragment_pixels = max(
    1,
    int(round(MIN_CROWN_AREA_M2 / work_resolution**2)),
)

canopy_mask = morphology.remove_small_objects(
    canopy_mask,
    min_size=minimum_fragment_pixels,
)
canopy_mask = morphology.binary_closing(
    canopy_mask,
    morphology.disk(1),
)

CANOPY_MASK_PATH = OUTPUT_DIR / "non_musanga_canopy_mask.tif"

mask_profile = {
    **profile,
    "dtype": "uint8",
    "nodata": 255,
}

with rasterio.open(CANOPY_MASK_PATH, "w", **mask_profile) as dst:
    dst.write(canopy_mask.astype(np.uint8), 1)

print("Canopy area:", canopy_mask.sum() * work_resolution**2, "m²")
print("Saved:", CANOPY_MASK_PATH)


## BLOCK 05 — Crown-top detection and RGB–CHM watershed

In [ ]:

# ======================================================================================
# BLOCK 05 — CROWN-TOP DETECTION AND RGB–CHM WATERSHED
# ======================================================================================

sigma_px = max(0.5, CHM_SMOOTHING_SIGMA_M / work_resolution)

smoothed_chm = ndi.gaussian_filter(
    np.where(chm_valid, chm, 0),
    sigma=sigma_px,
)

minimum_distance_px = max(
    1,
    int(round(MIN_CROWN_SPACING_M / work_resolution)),
)

peak_coordinates = peak_local_max(
    smoothed_chm,
    min_distance=minimum_distance_px,
    threshold_abs=MIN_PEAK_HEIGHT_M,
    labels=canopy_mask.astype(np.uint8),
    exclude_border=False,
)

markers = np.zeros(canopy_mask.shape, dtype=np.int32)

for marker_id, (row, col) in enumerate(peak_coordinates, start=1):
    markers[row, col] = marker_id

markers = ndi.grey_dilation(markers, size=(3, 3))
markers, n_markers = ndi.label(markers > 0)

print("Crown-top markers:", n_markers)

gray = (
    0.2989 * rgb_hwc[..., 0]
    + 0.5870 * rgb_hwc[..., 1]
    + 0.1140 * rgb_hwc[..., 2]
)
rgb_gradient = filters.sobel(gray)

chm_normalized = np.zeros_like(smoothed_chm, dtype=np.float32)
values = smoothed_chm[canopy_mask]

if len(values):
    lo, hi = np.percentile(values, [2, 98])
    if hi > lo:
        chm_normalized[canopy_mask] = np.clip(
            (smoothed_chm[canopy_mask] - lo) / (hi - lo),
            0,
            1,
        )

# High CHM pixels become basins; RGB edges increase boundary cost.
segmentation_surface = (
    0.70 * (1.0 - chm_normalized)
    + 0.30 * rgb_gradient
)

crown_labels = segmentation.watershed(
    segmentation_surface,
    markers=markers,
    mask=canopy_mask,
    compactness=0.001,
)

LABEL_RASTER_PATH = OUTPUT_DIR / "non_musanga_crown_labels.tif"

label_profile = {
    **profile,
    "dtype": "int32",
    "nodata": 0,
}

with rasterio.open(LABEL_RASTER_PATH, "w", **label_profile) as dst:
    dst.write(crown_labels.astype(np.int32), 1)

print("Raw crown segments:", int(crown_labels.max()))
print("Saved:", LABEL_RASTER_PATH)


## BLOCK 06 — Polygonize and filter crown segments

In [ ]:

# ======================================================================================
# BLOCK 06 — POLYGONIZE AND FILTER CROWN SEGMENTS
# ======================================================================================

records = []

for geom_mapping, value in tqdm(
    shapes(
        crown_labels.astype(np.int32),
        mask=crown_labels > 0,
        transform=work_transform,
    ),
    desc="Polygonizing",
):
    crown_id = int(value)
    if crown_id <= 0:
        continue

    geom = shape(geom_mapping)
    area = float(geom.area)

    if area < MIN_CROWN_AREA_M2 or area > MAX_CROWN_AREA_M2:
        continue

    pixels = crown_labels == crown_id
    heights = chm[pixels & chm_valid]

    if len(heights) == 0:
        continue

    mean_h = float(np.mean(heights))
    max_h = float(np.max(heights))

    if mean_h < MIN_TREE_HEIGHT_M:
        continue

    records.append({
        "crown_id": crown_id,
        "area_m2": area,
        "mean_h_m": mean_h,
        "max_h_m": max_h,
        "geometry": geom,
    })

crowns = gpd.GeoDataFrame(
    records,
    geometry="geometry",
    crs=rgb_crs,
)

if crowns.empty:
    raise RuntimeError(
        "No crown polygons survived filtering. "
        "Relax the height, spacing, or area settings."
    )

crowns["geometry"] = (
    crowns.geometry
    .buffer(0)
    .simplify(work_resolution * 0.5, preserve_topology=True)
)

crowns = crowns[
    ~crowns.geometry.is_empty
].reset_index(drop=True)

crowns["crown_id"] = np.arange(1, len(crowns) + 1)

GPKG_PATH = OUTPUT_DIR / "non_musanga_tree_crowns.gpkg"
SHP_PATH = OUTPUT_DIR / "non_musanga_tree_crowns.shp"

crowns.to_file(GPKG_PATH, layer="crowns", driver="GPKG")
crowns.to_file(SHP_PATH, driver="ESRI Shapefile")

print("Final crowns:", len(crowns))
print("Median area:", crowns.area.median(), "m²")
print("Median mean height:", crowns["mean_h_m"].median(), "m")
print("Saved:", GPKG_PATH)


## BLOCK 07 — Visual quality-control examples

In [ ]:

# ======================================================================================
# BLOCK 07 — VISUAL QUALITY-CONTROL EXAMPLES
# ======================================================================================

FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

N_EXAMPLES = 6
WINDOW_SIZE_M = 80.0
rng = np.random.default_rng(42)

selected = rng.choice(
    len(crowns),
    size=min(N_EXAMPLES, len(crowns)),
    replace=False,
)

for example_number, crown_index in enumerate(selected, start=1):

    centroid = crowns.iloc[int(crown_index)].geometry.centroid
    half = WINDOW_SIZE_M / 2
    bounds = (
        centroid.x - half,
        centroid.y - half,
        centroid.x + half,
        centroid.y + half,
    )

    min_col, min_row = ~work_transform * (bounds[0], bounds[3])
    max_col, max_row = ~work_transform * (bounds[2], bounds[1])

    r0 = max(0, int(np.floor(min_row)))
    r1 = min(work_height, int(np.ceil(max_row)))
    c0 = max(0, int(np.floor(min_col)))
    c1 = min(work_width, int(np.ceil(max_col)))

    rgb_patch = rgb_hwc[r0:r1, c0:c1]
    chm_patch = chm[r0:r1, c0:c1]

    patch_transform = work_transform * rasterio.Affine.translation(c0, r0)
    patch_bounds = rasterio.transform.array_bounds(
        r1 - r0,
        c1 - c0,
        patch_transform,
    )
    patch_box = box(*patch_bounds)
    patch_crowns = crowns[crowns.intersects(patch_box)]

    figure, axes = plt.subplots(1, 3, figsize=(17, 6))

    axes[0].imshow(rgb_patch)
    axes[0].set_title("RGB")

    im = axes[1].imshow(
        np.where(chm_patch == -9999, np.nan, chm_patch),
        cmap="viridis",
    )
    axes[1].set_title("CHM")
    figure.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    axes[2].imshow(rgb_patch)

    # Convert polygon coordinates to patch pixel coordinates for display.
    for geom in patch_crowns.geometry:
        if geom.geom_type == "Polygon":
            geoms = [geom]
        else:
            geoms = list(geom.geoms)

        for part in geoms:
            x, y = part.exterior.xy
            cols, rows = ~patch_transform * (np.asarray(x), np.asarray(y))
            axes[2].plot(cols, rows, linewidth=1.0)

    axes[2].set_title("Delineated non-Musanga crowns")

    for axis in axes:
        axis.axis("off")

    figure.tight_layout()
    path = FIGURE_DIR / f"crown_example_{example_number:02d}.png"
    figure.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(figure)

print("Figures saved to:", FIGURE_DIR)


## BLOCK 08 — Optional evaluation against digitized crowns

In [ ]:

# ======================================================================================
# BLOCK 08 — OPTIONAL EVALUATION AGAINST DIGITIZED CROWNS
# ======================================================================================

if REFERENCE_CROWNS_PATH is None or not REFERENCE_CROWNS_PATH.exists():

    print("No reference crown dataset found; evaluation skipped.")

else:

    reference = gpd.read_file(REFERENCE_CROWNS_PATH)

    if reference.crs != crowns.crs:
        reference = reference.to_crs(crowns.crs)

    class_candidates = [
        "Clss_ID", "class_id", "reference_class", "class"
    ]
    class_column = next(
        (col for col in class_candidates if col in reference.columns),
        None,
    )

    if class_column is not None:
        reference = reference[
            reference[class_column] != 1
        ].copy()

    reference = reference[
        ~reference.geometry.is_empty
    ].reset_index(drop=True)

    reference["reference_id"] = np.arange(len(reference))

    candidates = gpd.sjoin(
        crowns[["crown_id", "geometry"]],
        reference[["reference_id", "geometry"]],
        how="inner",
        predicate="intersects",
    )

    rows = []

    for item in tqdm(
        candidates.itertuples(),
        total=len(candidates),
        desc="Computing IoU",
    ):
        pred = crowns.loc[
            crowns["crown_id"] == item.crown_id,
            "geometry",
        ].iloc[0]

        ref = reference.loc[
            reference["reference_id"] == item.reference_id,
            "geometry",
        ].iloc[0]

        intersection = pred.intersection(ref).area
        union = pred.union(ref).area
        iou = intersection / union if union > 0 else 0

        rows.append({
            "crown_id": item.crown_id,
            "reference_id": item.reference_id,
            "iou": iou,
        })

    matches = pd.DataFrame(rows)

    if matches.empty:
        print("No predicted/reference intersections found.")
    else:
        best = (
            matches.sort_values("iou", ascending=False)
            .drop_duplicates("reference_id")
        )

        summary = pd.DataFrame([{
            "n_reference_non_musanga": len(reference),
            "n_predicted_crowns": len(crowns),
            "detection_rate_iou_025": np.mean(best["iou"] >= 0.25),
            "detection_rate_iou_050": np.mean(best["iou"] >= 0.50),
            "median_best_iou": best["iou"].median(),
            "mean_best_iou": best["iou"].mean(),
        }])

        display(summary.style.format({
            "detection_rate_iou_025": "{:.3f}",
            "detection_rate_iou_050": "{:.3f}",
            "median_best_iou": "{:.3f}",
            "mean_best_iou": "{:.3f}",
        }))

        best.to_csv(
            OUTPUT_DIR / "crown_reference_best_matches.csv",
            index=False,
        )
        summary.to_csv(
            OUTPUT_DIR / "crown_delineation_evaluation.csv",
            index=False,
        )


## BLOCK 09 — Final summary

In [ ]:

# ======================================================================================
# BLOCK 09 — FINAL SUMMARY
# ======================================================================================

summary = {
    "rgb": str(RGB_PATH),
    "dsm": str(DSM_PATH),
    "dtm": str(DTM_PATH) if DTM_PATH else None,
    "musanga_mask": str(MUSANGA_MASK_PATH),
    "working_resolution_m": float(work_resolution),
    "n_non_musanga_crowns": int(len(crowns)),
    "median_crown_area_m2": float(crowns.area.median()),
    "median_mean_height_m": float(crowns["mean_h_m"].median()),
    "outputs": {
        "geopackage": str(GPKG_PATH),
        "shapefile": str(SHP_PATH),
        "label_raster": str(LABEL_RASTER_PATH),
        "canopy_mask": str(CANOPY_MASK_PATH),
        "chm": str(CHM_PATH),
        "figures": str(FIGURE_DIR),
    },
}

SUMMARY_PATH = OUTPUT_DIR / "crown_delineation_summary.json"
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))
print("\nSaved summary:", SUMMARY_PATH)
